# Spreadsheet Functions and Data Preparation

**STA 532 2.0 Spreadsheet Modelling** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta532.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [What a spreadsheet model represents](#theory-01)
- [Dataset structure](#theory-02)
- [Types and missing values](#theory-03)
- [References and formula propagation](#theory-04)
- [Logical rules and boundary cases](#theory-05)
- [Validation, cleaning and traceability](#theory-06)
- [Filtering and sorting](#theory-07)
- [Python correspondence and audit](#theory-08)

## Detailed theory · Spreadsheet models and reliable data preparation

<a id="theory-01"></a>

### What a spreadsheet model represents
A model connects explicit inputs, formulas and outputs to a stated question. Identify units, assumptions and the decision supported by the output. Separate raw observations, editable inputs and calculated results; repeated unexplained constants make a model hard to audit.

<a id="theory-02"></a>

### Dataset structure
Use one observation per row and one variable per column with stable headers. A cell should contain one interpretable value, rather than merged categories or a number with an embedded unit string. Keep an identifier distinct from a quantitative measurement. Dates must be actual dates rather than ambiguous text.

<a id="theory-03"></a>

### Types and missing values
Text, numeric values, dates and logical values behave differently in formulas and sorting. A blank, a formula returning an empty string, a zero and a missing-value code may be treated differently. AVERAGE normally ignores blanks rather than treating them as zero; COUNT and COUNTA count different things. Document missingness and denominators instead of relying on accidental defaults.

<a id="theory-04"></a>

### References and formula propagation
Relative references move when copied; absolute references fix row and column; mixed references fix one. Use a fixed input cell for a shared rate and row-specific references for observations. Verify copied formulas at the beginning, middle and end of the region. Sorting only one column detaches records and corrupts the dataset.

<a id="theory-05"></a>

### Logical rules and boundary cases
IF selects between outputs based on a condition. AND requires every condition and OR at least one. NOT reverses a logical condition. Parentheses express intended precedence. Distinguish > from ≥ and check values exactly at a cutoff. Nested rules should be exhaustive and ordered so earlier branches do not mask later ones.

<a id="theory-06"></a>

### Validation, cleaning and traceability
Use range and category validation, duplicate checks, date checks and cross-field logic. Record why a duplicate is genuine or erroneous before changing it. Protect raw observations and retain a correction log. Broad error suppression can conceal a defective formula; handle only the intended error condition.

<a id="theory-07"></a>

### Filtering and sorting
A filter hides records according to a rule; it does not necessarily change the range included by a formula. Some subtotal functions distinguish hidden rows from included rows. State whether calculations concern all data or the filtered subset and inspect visible totals.

<a id="theory-08"></a>

### Python correspondence and audit
Python comparisons return Boolean values; conditional expressions or array operations implement the same rule. A code result is only useful if it preserves the same units, missing-value convention and row alignment as the spreadsheet. Check a few records by hand and confirm aggregate totals before building a chart or dashboard.

## Concepts covered

**Course points:** Logical IF; summary measures; Analysis ToolPak equivalents; dashboards; outlier checks and correlation.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Spreadsheet preparation: audit rows and logical rules

### Question and assumptions

A small worksheet has numeric revenues, one missing cost, and a duplicate ID. Show each preparation decision explicitly.

### Procedure

1. Check IDs and missing values before calculating.
2. Calculate profit only when both inputs are observed.
3. State the IF rule and test boundary values.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Row | ID | Revenue | Cost | Profit | ID check |
| --- | --- | --- | --- | --- | --- |
| 1 | A | 100 | 70 | 30 | review duplicate |
| 2 | B | 250 | missing | not calculated | unique |
| 3 | A | 100 | 70 | 30 | review duplicate |
| 4 | C | 200 | 150 | 50 | unique |

| Rule | Input | Output |
| --- | --- | --- |
| IF profit >= 50 | 30 | Below target |
| IF profit >= 50 | 50 | At or above target |
| Missing cost | None | Investigate; do not substitute zero |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


data=[["A",100,70],["B",250,None],["A",100,70],["C",200,150]]
rows=[]
for i,(identifier,revenue,cost) in enumerate(data,1):
    profit=None if cost is None else revenue-cost
    rows.append([i,identifier,revenue,"missing" if cost is None else cost,
                 "not calculated" if profit is None else profit,
                 "review duplicate" if sum(r[0]==identifier for r in data)>1 else "unique"])
show_table(["Row","ID","Revenue","Cost","Profit","ID check"],rows)
show_table(["Rule","Input","Output"],[["IF profit >= 50","30","Below target"],["IF profit >= 50","50","At or above target"],["Missing cost","None","Investigate; do not substitute zero"]])


### Decision and interpretation

A duplicate ID is a reason to investigate, not automatic permission to delete a row. Missing cost is different from cost equal to zero.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
week=np.arange(1,53); sales=1200+8*week+rng.normal(0,150,52); sales[[10,37]]*=2.3
status=np.where(sales>=1600,'target met','review'); q1,med,q3=np.quantile(sales,[.25,.5,.75]); i=q3-q1
print('AVERAGE',sales.mean().round(1),'MEDIAN',med.round(1),'STDEV.S',sales.std(ddof=1).round(1))
print('COUNTIF',np.sum(status=='target met'),'CORREL',np.corrcoef(week,sales)[0,1].round(3))
print('IQR flagged weeks',(np.where((sales<q1-1.5*i)|(sales>q3+1.5*i))[0]+1).tolist())
fig,ax=plt.subplots(1,2); ax[0].plot(week,sales); ax[1].boxplot(sales); plt.show()

### Practice set · Spreadsheet to story
1. Write Python equivalents for IF(sales≥1600), COUNTIF, AVERAGE, MEDIAN, and STDEV.S.
2. A 1.5×IQR rule flags a high sale. List two valid explanations besides a data-entry error.
3. Design a two-panel dashboard for weekly sales: choose one trend view and one distribution view.
4. Explain why source data should remain separate from formula-derived flags.

<details><summary>Answer sketches</summary>
1. np.where; np.sum(condition); np.mean; np.median; np.std(x,ddof=1). 2. A legitimate promotion, seasonal spike, or distinct customer segment. 3. Time plot plus histogram/boxplot. 4. Separation preserves provenance, enables review, and prevents accidental loss of the original measurement.
</details>

### Debrief
Check a flagged value against the source before modifying a dataset.

Record the formula, assumptions, and one follow-up question.